# GPR — Calculation

Fits the GPR model on the residual target, runs cross-validation, and saves all artifacts (model, splits, predictions, reconstructions, metrics) to `OUTPUT_FOLDER`. Plotting lives in `GPR_plotting.ipynb`.


In [1]:
import time
notebook_start = time.perf_counter()

import  os
import  json
import  pandas as pd
import  numpy as np
import  joblib
from    thermoift import print_model_metrics
from    thermoift.rng_utils import get_rng
from    sklearn.preprocessing import StandardScaler
from    sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from    sklearn.gaussian_process import GaussianProcessRegressor
from    sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from    sklearn.pipeline import Pipeline
from    sklearn.metrics import r2_score as _r2, mean_absolute_error as _mae
from    threadpoolctl import threadpool_info, threadpool_limits


In [2]:
OUTPUT_FOLDER           = "GPR_RESIDUAL_OUTPUTS"
SEED                    = 4555525
EXPERIMENT_MAX_SAMPLES  = 300  # Set to None to use the full dataset
RESTART_OPTIMIZER       = 1

# Stratified split/sampling settings
STRAT_BINS_SAMPLING     = 10    # T-P quantile bins used during stratified sampling
STRAT_BINS_SPLIT        = 4     # T-P quantile bins used for train/test/val split (q=4 -> 13 strata, safe down to 1000 samples)

# Cross-validation settings
RUN_CV                  = False  # set False to skip (~10 min on full dataset)
CV_FOLDS                = 5

CPU_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.environ.get("OMP_NUM_THREADS", "1")))
THREADPOOL_CONTROLLER = threadpool_limits(limits=CPU_THREADS)
print(f"Requested CPU threads: {CPU_THREADS}")
print(f"Active threadpools: {threadpool_info()}")

Requested CPU threads: 51
Active threadpools: [{'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 51, 'prefix': 'libscipy_openblas', 'filepath': '/home/darshan/A6/py_A6/lib/python3.12/site-packages/numpy.libs/libscipy_openblas64_-fdde5778.so', 'version': '0.3.30', 'threading_layer': 'pthreads', 'architecture': 'SkylakeX'}, {'user_api': 'blas', 'internal_api': 'openblas', 'num_threads': 51, 'prefix': 'libscipy_openblas', 'filepath': '/home/darshan/A6/py_A6/lib/python3.12/site-packages/scipy.libs/libscipy_openblas-b75cc656.so', 'version': '0.3.29.dev', 'threading_layer': 'pthreads', 'architecture': 'SkylakeX'}, {'user_api': 'openmp', 'internal_api': 'openmp', 'num_threads': 51, 'prefix': 'libgomp', 'filepath': '/home/darshan/A6/py_A6/lib/python3.12/site-packages/scikit_learn.libs/libgomp-a34b3233.so.1.0.0', 'version': None}]


In [3]:
# Parameters
OUTPUT_FOLDER = "SLURM_GPR_residual_5000"
SEED = 555555555
EXPERIMENT_MAX_SAMPLES = 5000
RESTART_OPTIMIZER = 10
STRAT_BINS_SAMPLING = 10
STRAT_BINS_SPLIT = 2
RUN_CV = True
CV_FOLDS = 5


In [4]:
df          = pd.read_csv("../CombinedDatasetSEC_A4.csv")
df.columns  = [col.strip().replace(" ", "_") for col in df.columns]

# Keep only rows needed to calculate gamma_cDFT and residual
df = df.dropna(subset=["gamma_wsd","gamma_cDFT_minus_wsd_uncorrected","gamma_wsd_UC"]).copy()

# Recalculate gamma_cDFT
df["gamma_cDFT_UC"] = df["gamma_wsd_UC"] + df["gamma_cDFT_minus_wsd_uncorrected"]
df["gamma_cDFT"]    = df["gamma_wsd"]    + df["gamma_cDFT_minus_wsd_corrected"]

# Residual
df["residual"]   = df["gamma_cDFT_UC"] - df["gamma_wsd_UC"]

tol = 1e-10
# Check gamma_cDFT_UC vs gamma_cDFT
diff_gamma = df["gamma_cDFT_UC"] - df["gamma_cDFT"]
mask_gamma = diff_gamma.abs() > tol

if mask_gamma.any():
    print(f"WARNING: {mask_gamma.sum()} rows have mismatched gamma_cDFT values.")

    print(df.loc[mask_gamma, [
        "source_id",
        "gamma_cDFT_UC",
        "gamma_cDFT"
    ]].assign(difference=diff_gamma[mask_gamma]))
else:
    print("All gamma_cDFT values match within tolerance.")

All gamma_cDFT values match within tolerance.


In [5]:
if EXPERIMENT_MAX_SAMPLES is not None:
    # Stratified sampling over T–P bins to guarantee coverage of rare regions
    df["_T_bin"]    = pd.qcut(df["T"], q=STRAT_BINS_SAMPLING, labels=False, duplicates="drop")
    df["_P_bin"]    = pd.qcut(df["P"], q=STRAT_BINS_SAMPLING, labels=False, duplicates="drop")
    df["_stratum"]  = df["_T_bin"].astype(str) + "_" + df["_P_bin"].astype(str)
    n_target        = min(EXPERIMENT_MAX_SAMPLES, len(df))
    df = (
        df.groupby("_stratum", group_keys=False)
          .apply(lambda g: g.sample(
              n=max(1, round(n_target * len(g) / len(df))),
              random_state=SEED,
          ))
          .sample(frac=1, random_state=SEED)  # shuffle
          .head(n_target)                     # exact cap
          .drop(columns=["_T_bin", "_P_bin", "_stratum"])
          .reset_index(drop=True)
    )

print(f"Total active samples: {len(df)}")
print(f"Experiment sample cap: {EXPERIMENT_MAX_SAMPLES}")
print(f"Residual statistics:")
print(df["residual"].describe())

Total active samples: 5000
Experiment sample cap: 5000
Residual statistics:
count    5000.000000
mean       -0.496698
std         0.581320
min        -4.986126
25%        -0.665218
50%        -0.280521
75%        -0.108202
max        -0.005577
Name: residual, dtype: float64


/tmp/ipykernel_1648304/3893283935.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


In [6]:
target          = "residual"
rng             = get_rng(seed=SEED)

z_columns       = [col for col in df.columns if col.startswith("z_")]
DROP_FEATURES   = ["z_oxygen"]
Z_non_zero      = [col for col in z_columns if (df[col] != 0).any() and col not in DROP_FEATURES]
features        = ["T", "P"] + Z_non_zero

print(f"Selected features: {features}")

X               = df[features]
y               = df[target]

# Stratified 70/15/15 split using T-P bins
_T_bin   = pd.qcut(X["T"], q=STRAT_BINS_SPLIT, labels=False, duplicates="drop")
_P_bin   = pd.qcut(X["P"], q=STRAT_BINS_SPLIT, labels=False, duplicates="drop")
_stratum = _T_bin.astype(str) + "_" + _P_bin.astype(str)

X_train, X_temp, y_train, y_temp, s_train, s_temp = train_test_split(
    X, y, _stratum, test_size=0.30, random_state=SEED, stratify=_stratum
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=s_temp
)

print(f"Training samples:   {X_train.shape[0]}")
print(f"Testing samples:      {X_test.shape[0]}")
print(f"Validation samples:   {X_val.shape[0]}")

Selected features: ['T', 'P', 'z_argon', 'z_carbon_dioxide', 'z_carbon_monoxide', 'z_hydrogen', 'z_hydrogen_sulfide', 'z_methane', 'z_nitrogen']
Training samples:   3500
Testing samples:      750
Validation samples:   750


In [7]:
# ── Baseline over/underprediction inspection ─────────────────────────────────
# Δγ < 0  →  baseline OVERpredicts  γ_cDFT  (most negative residuals)
# Δγ > 0  →  baseline UNDERpredicts γ_cDFT  (most positive residuals)
N = 10

z_cols    = [c for c in features if c.startswith("z_")]
z_non_co2 = [c for c in z_cols if c != "z_carbon_dioxide"]

inspect = df[features + [target]].copy()
inspect["source_id"]        = df["source_id"].values
inspect["dominant_species"] = inspect[z_non_co2].idxmax(axis=1).str.replace("z_", "")

cols_out = ["source_id", "dominant_species", "T", "P", target]

print(f"Top-{N} BASELINE OVERPREDICTIONS  (Δγ most negative — baseline too high):")
print(inspect.nsmallest(N, target)[cols_out].to_string(index=False))

print(f"Top-{N} BASELINE UNDERPREDICTIONS (Δγ most positive — baseline too low):")
print(inspect.nlargest(N, target)[cols_out].to_string(index=False))


Top-10 BASELINE OVERPREDICTIONS  (Δγ most negative — baseline too high):
 source_id dominant_species          T          P  residual
        21         hydrogen 200.000000 161.246559 -4.986126
        79         hydrogen 200.000000 135.764475 -4.614813
        73         nitrogen 202.009685  32.403171 -4.178747
        21         hydrogen 200.000000 121.621627 -4.152399
        98         nitrogen 202.028691  39.945444 -3.959027
        42         hydrogen 202.058557 117.464549 -3.884942
        79         hydrogen 200.000000 102.514849 -3.862751
        47         nitrogen 200.000000  27.737861 -3.702991
        63            argon 200.000000  53.038989 -3.649333
        67          methane 200.000000  24.818231 -3.627743
Top-10 BASELINE UNDERPREDICTIONS (Δγ most positive — baseline too low):
 source_id dominant_species          T         P  residual
        99         hydrogen 212.450083  4.887612 -0.005577
        99         hydrogen 220.750138  6.942795 -0.006983
        99        

In [8]:
df["gamma_base"] = df["gamma_wsd_UC"]

print(df[["gamma_cDFT", "gamma_base", "residual"]].isna().sum())
print("Any NaN in X:", X.isna().any().any())
print("Any NaN in y:", y.isna().any())

gamma_cDFT    0
gamma_base    0
residual      0
dtype: int64
Any NaN in X: False
Any NaN in y: False


In [9]:
# Kernel 
#Choice1: const * RBF + WhiteKernel
n_features = X_train.shape[1]
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3)) *
    RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e3)) +
    WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e0))
)

# Pipeline
gpr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr", GaussianProcessRegressor(
        kernel      = kernel,
        alpha       = 0.0,
        normalize_y = True,
        n_restarts_optimizer = RESTART_OPTIMIZER,
        random_state         = SEED,
    ))
]) 

# Fit
gpr_model.fit(X_train, y_train)

# Predictions
y_train_pred, y_train_std   = gpr_model.predict(X_train, return_std=True)
y_val_pred, y_val_std       = gpr_model.predict(X_val, return_std=True)
y_test_pred, y_test_std     = gpr_model.predict(X_test, return_std=True)

# Metrics
metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="mN/m", y_val=y_val, y_val_pred=y_val_pred)
metrics["train_mae_mNm"] = float(_mae(y_train, y_train_pred))
metrics["test_mae_mNm"]  = float(_mae(y_test,  y_test_pred))
metrics["val_mae_mNm"]   = float(_mae(y_val,   y_val_pred))
print(f"Residual MAE [mN/m] — train: {metrics['train_mae_mNm']:.6f} "
      f"| test: {metrics['test_mae_mNm']:.6f} "
      f"| val: {metrics['val_mae_mNm']:.6f}")

Model Performance for residual

Training Set:
  R²:   0.999850
  RMSE: 0.007240 mN/m
  MAE:  0.002897 mN/m

Test Set:
  R²:   0.999297
  RMSE: 0.015839 mN/m
  MAE:  0.005643 mN/m

Validation Set:
  R²:   0.999120
  RMSE: 0.015363 mN/m
  MAE:  0.005312 mN/m
Residual MAE [mN/m] — train: 0.002897 | test: 0.005643 | val: 0.005312


In [10]:
# 5-fold stratified cross-validation (same T-P binning used during sampling)
if RUN_CV:
    # Re-create stratum labels from X so each fold mirrors the T-P coverage
    _T_bin_cv = pd.qcut(X["T"], q=10, labels=False, duplicates="drop")
    _P_bin_cv = pd.qcut(X["P"], q=10, labels=False, duplicates="drop")
    _strata   = _T_bin_cv.astype(str) + "_" + _P_bin_cv.astype(str)

    _skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    cv_r2_scores   = cross_val_score(gpr_model, X, y, cv=_skf.split(X, _strata), scoring='r2')
    cv_rmse_scores = -cross_val_score(gpr_model, X, y, cv=_skf.split(X, _strata), scoring='neg_root_mean_squared_error')
    cv_mae_scores  = -cross_val_score(gpr_model, X, y, cv=_skf.split(X, _strata), scoring='neg_mean_absolute_error')

    print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
    print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
    print(f"Cross-Validation RMSE Scores: {cv_rmse_scores}")
    print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
    print(f"Cross-Validation MAE Scores:  {cv_mae_scores}")
    print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")
else:
    cv_r2_scores   = np.array([])
    cv_rmse_scores = np.array([])
    cv_mae_scores  = np.array([])
    print("CV skipped (RUN_CV=False).")

/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Cross-Validation R² Scores:   [0.99833141 0.99901542 0.99532812 0.99881182 0.97771022]
Mean CV R²:   0.993839 (+/- 0.016348)
Cross-Validation RMSE Scores: [0.02298591 0.01895659 0.04002591 0.01943971 0.08789433]
Mean CV RMSE: 0.037860 (+/- 0.052353)
Cross-Validation MAE Scores:  [0.00458046 0.00452642 0.00446737 0.00455912 0.00717674]
Mean CV MAE:  0.005062 (+/- 0.002116)


In [11]:
# GPR ARD feature importance: inverse length scales from the RBF kernel
# Kernel structure: (ConstantKernel * RBF) + WhiteKernel -> k1.k2.length_scale
_length_scales       = gpr_model.named_steps["gpr"].kernel_.k1.k2.length_scale
_feature_importances = 1.0 / _length_scales
_feature_importances = _feature_importances / _feature_importances.sum()  # normalize


In [12]:
# ── Validation outlier inspection ────────────────────────────────────────────
N_OUTLIERS = 10

z_cols    = [c for c in features if c.startswith("z_")]
z_non_co2 = [c for c in z_cols if c != "z_carbon_dioxide"]

val_df = X_val.copy()
val_df["y_true"]           = y_val.values
val_df["y_pred"]           = y_val_pred
val_df["error"]            = y_val_pred - y_val.values
val_df["abs_err"]          = val_df["error"].abs()
val_df["source_id"]        = df.loc[val_df.index, "source_id"].values
val_df["dominant_species"] = val_df[z_non_co2].idxmax(axis=1).str.replace("z_", "")

outliers = val_df.nlargest(N_OUTLIERS, "abs_err")[
    ["source_id", "dominant_species", "T", "P", "y_true", "y_pred", "error"]
]

print(f"Top-{N_OUTLIERS} validation outliers (sorted by |error|):")
print(outliers.to_string(index=False))


Top-10 validation outliers (sorted by |error|):
 source_id dominant_species          T         P    y_true    y_pred     error
        99         hydrogen 208.300055 41.132062 -1.220417 -1.422101 -0.201684
         6  carbon_monoxide 200.000000 21.845592 -1.090561 -1.256502 -0.165941
        38         hydrogen 202.057902 45.148910 -2.254577 -2.373891 -0.119315
        64 hydrogen_sulfide 206.184150 25.367023 -1.760213 -1.867805 -0.107592
        14          methane 200.000000  4.324204 -0.350606 -0.252457  0.098150
        33         hydrogen 204.162333 22.067727 -0.833266 -0.922944 -0.089677
        33         hydrogen 200.000000 16.084434 -0.682590 -0.764505 -0.081915
        88  carbon_monoxide 202.036217 14.640967 -1.221069 -1.301074 -0.080005
        85 hydrogen_sulfide 200.000000  4.545444 -0.682329 -0.609462  0.072867
        80         hydrogen 200.000000 50.249522 -2.680537 -2.745608 -0.065072


In [13]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
model_path = os.path.join(OUTPUT_FOLDER, "GPR_residual_model.joblib")
joblib.dump(gpr_model, model_path)
print(f"Model saved to: {model_path}")

Model saved to: SLURM_GPR_residual_5000/GPR_residual_model.joblib


In [14]:
# Reconstruct full gamma_cDFT = gamma_base + residual_pred

gamma_base_train = df.loc[X_train.index, "gamma_base"].values
gamma_base_test  = df.loc[X_test.index,  "gamma_base"].values
gamma_base_val   = df.loc[X_val.index,   "gamma_base"].values

gamma_cDFT_train = df.loc[X_train.index, "gamma_cDFT"].values
gamma_cDFT_test  = df.loc[X_test.index,  "gamma_cDFT"].values
gamma_cDFT_val   = df.loc[X_val.index,   "gamma_cDFT"].values

gamma_pred_train = gamma_base_train + y_train_pred
gamma_pred_test  = gamma_base_test  + y_test_pred
gamma_pred_val   = gamma_base_val   + y_val_pred

# std carries through the additive reconstruction unchanged
gamma_cDFT_r2_train  = _r2(gamma_cDFT_train, gamma_pred_train)
gamma_cDFT_r2_test   = _r2(gamma_cDFT_test,  gamma_pred_test)
gamma_cDFT_r2_val    = _r2(gamma_cDFT_val,   gamma_pred_val)
gamma_cDFT_mae_train = _mae(gamma_cDFT_train, gamma_pred_train)
gamma_cDFT_mae_test  = _mae(gamma_cDFT_test,  gamma_pred_test)
gamma_cDFT_mae_val   = _mae(gamma_cDFT_val,   gamma_pred_val)

print(f"Reconstructed gamma_cDFT  R² — train: {gamma_cDFT_r2_train:.4f} "
      f"| test: {gamma_cDFT_r2_test:.4f} "
      f"| val:  {gamma_cDFT_r2_val:.4f}")
print(f"Reconstructed gamma_cDFT MAE [mN/m] — train: {gamma_cDFT_mae_train:.6f} "
      f"| test: {gamma_cDFT_mae_test:.6f} "
      f"| val: {gamma_cDFT_mae_val:.6f}")

Reconstructed gamma_cDFT  R² — train: 1.0000 | test: 1.0000 | val:  1.0000
Reconstructed gamma_cDFT MAE [mN/m] — train: 0.002897 | test: 0.005643 | val: 0.005312


## Save artifacts for plotting


In [15]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 1. Full (post-sampling) dataframe — feeds correlation heatmap and EDA scatter plots.
df_path = os.path.join(OUTPUT_FOLDER, "GPR_df_full.parquet")
df.to_parquet(df_path)

# 2. Split indices into the saved df (preserve original df indices for downstream joins).
splits_path = os.path.join(OUTPUT_FOLDER, "GPR_splits.npz")
np.savez(
    splits_path,
    train_idx = X_train.index.to_numpy(),
    test_idx  = X_test.index.to_numpy(),
    val_idx   = X_val.index.to_numpy(),
)

# 3. Predictions + predictive std for every split.
preds_path = os.path.join(OUTPUT_FOLDER, "GPR_predictions.npz")
np.savez(
    preds_path,
    y_train_pred = y_train_pred, y_train_std = y_train_std,
    y_test_pred  = y_test_pred,  y_test_std  = y_test_std,
    y_val_pred   = y_val_pred,   y_val_std   = y_val_std,
)

# 4. Reconstructed gamma_cDFT arrays.
recon_path = os.path.join(OUTPUT_FOLDER, "GPR_gamma_reconstructed.npz")
np.savez(
    recon_path,
    gamma_cDFT_train = gamma_cDFT_train, gamma_pred_train = gamma_pred_train,
    gamma_cDFT_test  = gamma_cDFT_test,  gamma_pred_test  = gamma_pred_test,
    gamma_cDFT_val   = gamma_cDFT_val,   gamma_pred_val   = gamma_pred_val,
)

# 5. ARD feature importances.
fi_path = os.path.join(OUTPUT_FOLDER, "GPR_feature_importances.npy")
np.save(fi_path, _feature_importances)

# 6. Lightweight metadata (features, target, output folder) so the plotting
#    notebook does not need to re-derive any column lists.
meta_path = os.path.join(OUTPUT_FOLDER, "GPR_artifacts_meta.json")
with open(meta_path, "w") as f:
    json.dump({
        "features":      features,
        "target":        target,
        "output_folder": OUTPUT_FOLDER,
        "seed":          SEED,
    }, f, indent=2)

for p in [df_path, splits_path, preds_path, recon_path, fi_path, meta_path, model_path]:
    print(f"saved: {p}")


saved: SLURM_GPR_residual_5000/GPR_df_full.parquet
saved: SLURM_GPR_residual_5000/GPR_splits.npz
saved: SLURM_GPR_residual_5000/GPR_predictions.npz
saved: SLURM_GPR_residual_5000/GPR_gamma_reconstructed.npz
saved: SLURM_GPR_residual_5000/GPR_feature_importances.npy
saved: SLURM_GPR_residual_5000/GPR_artifacts_meta.json
saved: SLURM_GPR_residual_5000/GPR_residual_model.joblib


In [16]:
# Collect and save all metrics
if RUN_CV:
    metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
    metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
    metrics["cv_r2_std"]      = float(cv_r2_scores.std())
    metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
    metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
    metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
    metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
    metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
    metrics["cv_mae_std"]     = float(cv_mae_scores.std())
else:
    metrics["cv_r2_scores"]   = []
    metrics["cv_r2_mean"]     = None
    metrics["cv_r2_std"]      = None
    metrics["cv_rmse_scores"] = []
    metrics["cv_rmse_mean"]   = None
    metrics["cv_rmse_std"]    = None
    metrics["cv_mae_scores"]  = []
    metrics["cv_mae_mean"]    = None
    metrics["cv_mae_std"]     = None

metrics["gamma_cDFT_r2_train"]      = float(gamma_cDFT_r2_train)
metrics["gamma_cDFT_r2_test"]       = float(gamma_cDFT_r2_test)
metrics["gamma_cDFT_r2_val"]        = float(gamma_cDFT_r2_val)
metrics["gamma_cDFT_mae_train_mNm"] = float(gamma_cDFT_mae_train)
metrics["gamma_cDFT_mae_test_mNm"]  = float(gamma_cDFT_mae_test)
metrics["gamma_cDFT_mae_val_mNm"]   = float(gamma_cDFT_mae_val)

metrics["model"]                = "GPR_RBF"
metrics["kernel"]               = str(gpr_model.named_steps["gpr"].kernel_)
metrics["alpha"]                = gpr_model.named_steps["gpr"].alpha
metrics["n_restarts_optimizer"] = gpr_model.named_steps["gpr"].n_restarts_optimizer
metrics["features"]             = features
metrics["target"]               = target
metrics["seed"]                 = SEED

metrics_path = os.path.join(OUTPUT_FOLDER, f"GPR_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print(f"Metrics saved to: {metrics_path}")


Metrics saved to: SLURM_GPR_residual_5000/GPR_residual_metrics.json


In [17]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 438.33 minutes
